In [1]:
!pip install geemap --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 9.7 MB/s eta 0:00:00


In [101]:
# Import necessary libraries
import ee
from google.colab import userdata
import geemap

# Get Earth Engine project name from Colab secrets
EE_PROJECT_NAME = userdata.get('EE_PROJECT_NAME')

# Authenticate and initialize Earth Engine
ee.Authenticate()
ee.Initialize(project=EE_PROJECT_NAME)

# Define Kenya boundary from FAO GAUL dataset
kenya = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM0_NAME', 'Kenya'))
counties = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM0_NAME', 'Kenya'))

# Select a county (e.g., Nakuru) as region of interest (ROI)
roi = counties.filter(ee.Filter.eq('ADM2_NAME', 'Nakuru'))

# Load JRC Global Surface Water
# Use "occurrence" band to identify permanent water bodies
jrc_water = ee.Image('JRC/GSW1_3/GlobalSurfaceWater').select('occurrence')
# Define permanent water as pixels with water occurrence > 90%
permanent_water = jrc_water.gt(50).selfMask()

#Create a binary land mask by inverting permanent water
land_mask = jrc_water.unmask(0).eq(0).selfMask()

# Load ESA WorldCover 2020 and extract cropland class (code 40)
esa = ee.ImageCollection('ESA/WorldCover/v100').first().select('Map')
esa_crop = esa.eq(40)

# Load GLC_FCS30D for 2020 and extract cropland class (code 10)
glc = ee.ImageCollection("projects/sat-io/open-datasets/FROM-GLC10").mosaic()
glc_crop = glc.eq(10)

# Load Dynamic World crops layer and mask high probability (>0.5) crop pixels
dynamic_world = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1') \
    .filterDate('2020-01-01', '2020-12-31') \
    .select('crops') \
    .mode()

# Threshold Dynamic World crop probability
dw_crop = dynamic_world.updateMask(dynamic_world.gt(0.5)).gt(0).selfMask()

# Combine agreement from all three crop masks
glc_crop_unmask = glc_crop.unmask(0)
esa_crop_unmask = esa_crop.unmask(0)
dw_crop_unmask = dw_crop.unmask(0)

agreement = esa_crop_unmask.add(glc_crop_unmask).add(dw_crop_unmask)
crop_agreement = agreement.eq(3)  # At least 3-way agreement for high-confidence cropland

# Clip the result to ROI and mask
crop_image = crop_agreement.updateMask(crop_agreement).clip(roi).rename('cropland')

# Define non-cropland where all three agree it's not crop
non_crop_image = agreement.eq(0).selfMask().clip(roi)
# Apply the non-water mask to your non_crop_image
non_crop_image = non_crop_image.updateMask(land_mask).rename('non-cropland')

In [105]:
# Stratified sampling of crop vs. non-crop
n_samples = 50000
sampled_crop_labels = crop_image.stratifiedSample(
    numPoints=n_samples // 2,
    classBand='cropland',
    region=roi.geometry(),
    scale=10,
    seed=42,
    geometries=True
)

# Sample non-crop points with label 0
sampled_non_crop_labels = non_crop_image.stratifiedSample(
    numPoints=n_samples // 2,
    classBand='non-cropland',
    region=roi.geometry(),
    scale=10,
    seed=43,
    geometries=True
)
sampled_crop_labels = sampled_crop_labels.map(lambda f: f.set('label', 1))
sampled_non_crop_labels = sampled_non_crop_labels.map(lambda f: f.set('label', 0))

# Merge crop and non-crop points
sampled_labels = sampled_crop_labels.merge(sampled_non_crop_labels)

In [106]:
Map = geemap.Map()
Map.add_basemap('HYBRID')
Map.addLayer(kenya, {}, 'Kenya')
Map.addLayer(roi, {}, 'Nakuru')
Map.centerObject(roi, 8)
#Map.addLayer(permanent_water, {'palette': 'blue'}, 'Permanent Water')
#Map.addLayer(land_mask, {'palette': 'green'}, 'Land Mask')
Map.addLayer(crop_image.visualize(**{'palette': ['#f096ff']}), {}, 'Crop')
Map.addLayer(non_crop_image.visualize(**{'palette': ['#b4b4b4']}), {}, 'Non-Crop-no-water')
# Filter out features with null geometries
#sampled_labels_filtered = sampled_labels.filter(ee.Filter.notNull(['geometry']))

Map.addLayer(sampled_labels.filter(ee.Filter.eq('cropland', 1)), {'color': 'ffff64'}, 'Crop Points')
Map.addLayer(sampled_labels.filter(ee.Filter.eq('non-cropland', 1)), {'color': '#a6a6a6'}, 'Non-Crop Points')

Map

Map(center=[-0.46101926483462224, 36.07982194698872], controls=(WidgetControl(options=['position', 'transparen…

In [ ]:
# Export cropland agreement map as GeoTIFF to Google Drive
export_task = ee.batch.Export.image.toDrive(
    image=crop_image,
    description='final_crop_nakuru_export',
    folder='gee_exports',
    fileNamePrefix='final_crop_nakuru',
    region=roi.geometry(),
    scale=10,
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)
export_task.start()

In [11]:
# Function to compute statistics for a given band
def compute_time_series_stats(collection, band):
    reducers = ee.Reducer.min()\
        .combine(ee.Reducer.max(), '', True)\
        .combine(ee.Reducer.mean(), '', True)\
        .combine(ee.Reducer.median(), '', True)\
        .combine(ee.Reducer.stdDev(), '', True)\
        .combine(ee.Reducer.skew(), '', True)\
        .combine(ee.Reducer.kurtosis(), '', True)

    return collection.select(band).reduce(reducers).rename([
        f'{band}_min', f'{band}_max', f'{band}_mean',
        f'{band}_median', f'{band}_stdDev', f'{band}_skew',
        f'{band}_kurtosis'
    ])

In [10]:
# Function to compute Bare Soil Index (BSI)
def calculate_bsi(sentinel2_image):
    bsi = sentinel2_image.expression(
        '((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))', {
            'SWIR1': sentinel2_image.select('B11'),
            'RED': sentinel2_image.select('B4'),
            'NIR': sentinel2_image.select('B8'),
            'BLUE': sentinel2_image.select('B2')
        }).rename('BSI')
    return bsi

# Function to compute indices and GLCM texture features
def add_all_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    ndwi = image.normalizedDifference(['B8', 'B11']).rename('NDWI')
    ndre = image.normalizedDifference(['B8', 'B5']).rename('NDRE')
    gndwi = image.normalizedDifference(['B3', 'B11']).rename('GNDWI')
    rendvi = image.normalizedDifference(['B7', 'B4']).rename('RENDVI')
    gndvi = image.normalizedDifference(['B8', 'B3']).rename('GNDVI')
    bsi = calculate_bsi(image)
    msr = image.expression('((NIR / RED - 1) / sqrt(NIR / RED + 1))', {
        'NIR': image.select('B8'),
        'RED': image.select('B4')
    }).rename('MSR')
    savi = image.expression('((NIR - RED) / (NIR + RED + 0.5)) * 1.5', {
        'NIR': image.select('B8'),
        'RED': image.select('B4')
    }).rename('SAVI')


    # GLCM texture from NDVI
    ndvi_8bit = ndvi.multiply(255).toUint8()
    glcm = ndvi_8bit.glcmTexture(3)
    return image.addBands([ndvi, ndwi, ndre, gndwi, rendvi, gndvi, bsi, msr, savi,
                           glcm.select('NDVI_asm').rename('GLCM_asm'),
                           glcm.select('NDVI_contrast').rename('GLCM_contrast'),
                           glcm.select('NDVI_corr').rename('GLCM_corr'),
                           glcm.select('NDVI_var').rename('GLCM_var')])

In [12]:
# Process Sentinel-2 imagery and apply indices
s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterDate('2020-03-01', '2020-11-01') \
    .filterBounds(kenya) \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(add_all_indices)

gndvi_stats = compute_time_series_stats(s2, 'GNDVI')
gndwi_stats = compute_time_series_stats(s2, 'GNDWI')
rendvi_stats = compute_time_series_stats(s2, 'RENDVI')
ndre_stats = compute_time_series_stats(s2, 'NDRE')
msr_stats = compute_time_series_stats(s2, 'MSR')
bsi_stats =  compute_time_series_stats(s2, 'BSI')


In [19]:
# Create a single composite for training
training_features = s2.select(['GLCM_corr', 'GLCM_var']).mosaic().addBands(gndvi_stats).addBands(gndwi_stats).addBands(rendvi_stats).addBands(ndre_stats).addBands(msr_stats).addBands(bsi_stats)

In [107]:
def sample_training_features(training_features, sampled_labels):
    # Sample the training data at the locations of sampled points
    sampled_training_features = training_features.sampleRegions(
        collection=sampled_labels,
        properties=['label'],  # Use the unified label field
        scale=10,
        geometries=True
    )
    return sampled_training_features

# Usage
sampled_training_features = sample_training_features(training_features, sampled_labels)


In [108]:
# Export to Google Drive
task = ee.batch.Export.table.toDrive(
    collection=sampled_training_features.filterBounds(roi.geometry()),
    description='feature_stats_training',
    fileFormat='CSV'
)

task.start()

In [ ]:
# Load DEM for terrain features (could add slope, aspect later)
srtm = ee.Image('USGS/SRTMGL1_003').clip(roi)